In [ ]:
#| default_exp misc.conv_decomposer

In [ ]:
#| include: false
from nbdev.showdoc import *

## Overview

The `Conv_Decomposer` class reduces model size and FLOPs by factorizing Conv2d layers into three smaller convolutions using Tucker decomposition. This is the Conv2d counterpart of `FC_Decomposer` (which uses SVD for Linear layers).

**How it works:** A Conv2d weight `[C_out, C_in, H, W]` is decomposed into:
1. `Conv2d(C_in, R_in, 1)` — pointwise input channel compression
2. `Conv2d(R_in, R_out, (H, W))` — spatial convolution at reduced rank
3. `Conv2d(R_out, C_out, 1)` — pointwise output channel expansion

### When to Use

| Scenario | Recommendation |
|----------|----------------|
| Large 3x3 or larger convolutions | **Highly recommended** — significant FLOP savings |
| 1x1 pointwise convolutions | Skipped automatically (already minimal) |
| Depthwise / grouped convolutions | Skipped (Tucker assumes standard convolution) |
| First layer (C_in=3) | Works but limited benefit |
| Post-training compression | Fine-tune after decomposition for best accuracy |

In [ ]:
#| export
import torch
import torch.nn as nn
import copy

In [ ]:
#| export
from fasterai.misc.fc_decomposer import _rank_from_energy, _should_decompose

def _unfold(tensor, mode):
    "Unfold a tensor along a mode into a matrix"
    return tensor.moveaxis(mode, 0).flatten(1)

def _partial_tucker(weight, ranks, n_iter=10, tol=1e-4):
    "Partial Tucker decomposition on modes [0, 1] via alternating SVD (HOOI)"
    U0 = torch.linalg.svd(_unfold(weight, 0), full_matrices=False)[0][:, :ranks[0]]
    U1 = torch.linalg.svd(_unfold(weight, 1), full_matrices=False)[0][:, :ranks[1]]

    for _ in range(n_iter):
        U0_prev, U1_prev = U0.clone(), U1.clone()
        proj = torch.einsum('oihw, or -> rihw', weight, U0)
        U1 = torch.linalg.svd(_unfold(proj, 1), full_matrices=False)[0][:, :ranks[1]]
        proj = torch.einsum('oihw, is -> oshw', weight, U1)
        U0 = torch.linalg.svd(_unfold(proj, 0), full_matrices=False)[0][:, :ranks[0]]
        if (U0 - U0_prev).norm() + (U1 - U1_prev).norm() < tol: break

    core = torch.einsum('oihw, or, is -> rshw', weight, U0, U1)
    return core, [U0, U1]

VALID_METHODS = frozenset({'tucker', 'svd', 'spatial', 'cp'})

class Conv_Decomposer:
    "Decompose Conv2d layers to reduce parameters and FLOPs"

    def __init__(self): pass

    def decompose(self,
                  model: nn.Module,                       # The model to decompose
                  percent_removed: float = 0.5,           # Fraction of rank to remove [0, 1)
                  method: str = 'tucker',                 # 'tucker', 'svd', 'spatial', or 'cp'
                  energy_threshold: float | None = None,  # Auto rank via energy retention (0-1)
                  layers: list[str] | None = None,        # Layer names to decompose (None = all eligible)
                  exclude: list[str] | None = None,       # Layer names to skip
                  n_iter: int = 10,                       # Max HOOI iterations (tucker only)
                  tol: float = 1e-4,                      # HOOI convergence tolerance (tucker only)
    ) -> nn.Module:
        "Decompose eligible Conv2d layers using the specified method."
        if method not in VALID_METHODS:
            raise ValueError(f"method must be one of {VALID_METHODS}, got {method!r}")
        if energy_threshold is None and not (0 <= percent_removed < 1):
            raise ValueError(f"percent_removed must be in range [0, 1), got {percent_removed}")
        if energy_threshold is not None and not (0 < energy_threshold <= 1):
            raise ValueError(f"energy_threshold must be in range (0, 1], got {energy_threshold}")

        decompose_fn = {'tucker': self.Tucker, 'svd': self.SVD,
                        'spatial': self.Spatial, 'cp': self.CP}[method]

        new_model = copy.deepcopy(model)
        for name, module in list(new_model.named_modules()):
            if (isinstance(module, nn.Conv2d) and module.groups == 1 
                and min(module.kernel_size) > 1
                and _should_decompose(name, layers, exclude)):
                parent_name, _, child_name = name.rpartition('.')
                parent = new_model.get_submodule(parent_name) if parent_name else new_model
                if method == 'tucker':
                    replacement = decompose_fn(module, percent_removed, energy_threshold, n_iter, tol)
                else:
                    replacement = decompose_fn(module, percent_removed, energy_threshold)
                setattr(parent, child_name, replacement)
        return new_model

    def SVD(self,
            layer: nn.Conv2d,                        # The Conv2d layer to decompose
            percent_removed: float = 0.5,            # Fraction of rank to remove
            energy_threshold: float | None = None,   # Auto rank via energy retention
    ) -> nn.Sequential:
        "SVD: 2 layers — spatial at reduced output rank + pointwise expansion"
        W = layer.weight.data
        C_out, C_in = W.shape[:2]
        K = layer.kernel_size

        W_2d = W.reshape(C_out, -1)
        U, S, Vh = torch.linalg.svd(W_2d, full_matrices=False)

        if energy_threshold is not None:
            R = _rank_from_energy(S, energy_threshold)
        else:
            R = max(1, int((1 - percent_removed) * min(C_out, C_in)))

        first = nn.Conv2d(C_in, R, K, stride=layer.stride,
                          padding=layer.padding, dilation=layer.dilation, bias=False)
        first.weight.data = (torch.diag(S[:R]) @ Vh[:R]).reshape(R, C_in, *K)

        last = nn.Conv2d(R, C_out, 1, bias=layer.bias is not None)
        last.weight.data = U[:, :R].unsqueeze(-1).unsqueeze(-1)
        if layer.bias is not None:
            last.bias.data = layer.bias.data

        return nn.Sequential(first, last)

    def Spatial(self,
                layer: nn.Conv2d,                       # The Conv2d layer to decompose
                percent_removed: float = 0.5,           # Fraction of spatial rank to remove
                energy_threshold: float | None = None,  # Auto rank via energy retention
    ) -> nn.Sequential:
        "Spatial separable: 2 layers — K×1 vertical + 1×K horizontal per filter"
        W = layer.weight.data
        C_out, C_in = W.shape[:2]
        Kh, Kw = layer.kernel_size

        # SVD on each filter's spatial matrix, average the rank across filters
        # Use first filter to determine rank
        S_sample = torch.linalg.svd(W[0, 0].reshape(Kh, Kw), full_matrices=False)[1]
        if energy_threshold is not None:
            R = _rank_from_energy(S_sample, energy_threshold)
        else:
            R = max(1, int((1 - percent_removed) * min(Kh, Kw)))

        # Vertical: Conv2d(C_in, C_out*R, Kh×1)
        # Horizontal: Conv2d(C_out*R, C_out, 1×Kw, groups=C_out)
        # Build weights by SVD of each filter's spatial component
        W_vert = torch.zeros(C_out * R, C_in, Kh, 1)
        W_horiz = torch.zeros(C_out, R, 1, Kw)

        for o in range(C_out):
            for i in range(C_in):
                U, S, Vh = torch.linalg.svd(W[o, i].reshape(Kh, Kw), full_matrices=False)
                for r in range(R):
                    W_vert[o * R + r, i, :, 0] = U[:, r] * S[r].sqrt()
                    W_horiz[o, r, 0, :] += Vh[r] * S[r].sqrt() / C_in

        vert = nn.Conv2d(C_in, C_out * R, (Kh, 1),
                         stride=(layer.stride[0], 1),
                         padding=(layer.padding[0], 0), bias=False)
        vert.weight.data = W_vert

        horiz = nn.Conv2d(C_out * R, C_out, (1, Kw), groups=C_out,
                          stride=(1, layer.stride[1]),
                          padding=(0, layer.padding[1]),
                          bias=layer.bias is not None)
        horiz.weight.data = W_horiz
        if layer.bias is not None:
            horiz.bias.data = layer.bias.data

        return nn.Sequential(vert, horiz)

    def CP(self,
           layer: nn.Conv2d,                        # The Conv2d layer to decompose
           percent_removed: float = 0.5,            # Fraction of rank to remove
           energy_threshold: float | None = None,   # Auto rank via energy retention
    ) -> nn.Sequential:
        "CP: 4 layers — pointwise compress + depthwise vertical + depthwise horizontal + pointwise expand"
        W = layer.weight.data
        C_out, C_in = W.shape[:2]
        Kh, Kw = layer.kernel_size

        # Determine rank from mode-0 unfolding
        S0 = torch.linalg.svd(_unfold(W, 0), full_matrices=False)[1]
        if energy_threshold is not None:
            R = _rank_from_energy(S0, energy_threshold)
        else:
            R = max(1, int((1 - percent_removed) * min(C_out, C_in)))

        # Full SVD on mode-0 unfolding: (C_out, C_in*Kh*Kw)
        W_2d = W.reshape(C_out, -1)
        U, S, Vh = torch.linalg.svd(W_2d, full_matrices=False)

        # Vh[:R] has shape (R, C_in*Kh*Kw) → reshape to (R, C_in, Kh, Kw)
        V_4d = Vh[:R].reshape(R, C_in, Kh, Kw)

        # Further decompose spatial dims of V_4d via SVD per rank component
        W_pw_in = torch.zeros(R, C_in, 1, 1)
        W_dw_v = torch.zeros(R, 1, Kh, 1)
        W_dw_h = torch.zeros(R, 1, 1, Kw)

        for r in range(R):
            # Average spatial component across input channels
            spatial_avg = V_4d[r].mean(dim=0)  # (Kh, Kw)
            u_s, s_s, vh_s = torch.linalg.svd(spatial_avg.reshape(Kh, Kw), full_matrices=False)
            W_dw_v[r, 0, :, 0] = u_s[:, 0] * s_s[0].sqrt()
            W_dw_h[r, 0, 0, :] = vh_s[0] * s_s[0].sqrt()
            # Channel component: norm of each input channel's contribution
            W_pw_in[r, :, 0, 0] = V_4d[r].pow(2).sum(dim=(1, 2)).sqrt() * S[r].sqrt()

        # Layer 1: pointwise input compression (C_in → R)
        pw_in = nn.Conv2d(C_in, R, 1, bias=False)
        pw_in.weight.data = W_pw_in

        # Layer 2: depthwise vertical (R → R, Kh×1)
        dw_v = nn.Conv2d(R, R, (Kh, 1), groups=R,
                         stride=(layer.stride[0], 1),
                         padding=(layer.padding[0], 0), bias=False)
        dw_v.weight.data = W_dw_v

        # Layer 3: depthwise horizontal (R → R, 1×Kw)
        dw_h = nn.Conv2d(R, R, (1, Kw), groups=R,
                         stride=(1, layer.stride[1]),
                         padding=(0, layer.padding[1]), bias=False)
        dw_h.weight.data = W_dw_h

        # Layer 4: pointwise output expansion (R → C_out)
        pw_out = nn.Conv2d(R, C_out, 1, bias=layer.bias is not None)
        pw_out.weight.data = (U[:, :R] * S[:R].sqrt().unsqueeze(0)).unsqueeze(-1).unsqueeze(-1)
        if layer.bias is not None:
            pw_out.bias.data = layer.bias.data

        return nn.Sequential(pw_in, dw_v, dw_h, pw_out)

    def Tucker(self,
               layer: nn.Conv2d,                        # The Conv2d layer to decompose
               percent_removed: float = 0.5,            # Fraction of rank to remove per mode
               energy_threshold: float | None = None,   # Auto rank via energy retention
               n_iter: int = 10,                        # Max HOOI iterations
               tol: float = 1e-4,                       # HOOI convergence tolerance
    ) -> nn.Sequential:
        "Tucker: 3 layers — pointwise compress + spatial + pointwise expand"
        W = layer.weight.data
        C_out, C_in = W.shape[:2]

        if energy_threshold is not None:
            S0 = torch.linalg.svd(_unfold(W, 0), full_matrices=False)[1]
            S1 = torch.linalg.svd(_unfold(W, 1), full_matrices=False)[1]
            R_out = _rank_from_energy(S0, energy_threshold)
            R_in = _rank_from_energy(S1, energy_threshold)
        else:
            R_out = max(1, int((1 - percent_removed) * C_out))
            R_in  = max(1, int((1 - percent_removed) * C_in))

        core, (U_out, U_in) = _partial_tucker(W, [R_out, R_in], n_iter=n_iter, tol=tol)

        first = nn.Conv2d(C_in, R_in, 1, bias=False)
        first.weight.data = U_in.t().unsqueeze(-1).unsqueeze(-1)

        middle = nn.Conv2d(R_in, R_out, layer.kernel_size, stride=layer.stride,
                           padding=layer.padding, dilation=layer.dilation, bias=False)
        middle.weight.data = core

        last = nn.Conv2d(R_out, C_out, 1, bias=layer.bias is not None)
        last.weight.data = U_out.unsqueeze(-1).unsqueeze(-1)
        if layer.bias is not None:
            last.bias.data = layer.bias.data

        return nn.Sequential(first, middle, last)

In [ ]:
show_doc(Conv_Decomposer)

In [ ]:
show_doc(Conv_Decomposer.decompose)

---

## Usage Example

```python
from fasterai.misc.conv_decomposer import Conv_Decomposer
from torchvision.models import resnet18

model = resnet18(pretrained=True)
decomposer = Conv_Decomposer()
compressed = decomposer.decompose(model, percent_removed=0.5)

# Check parameter reduction
orig = sum(p.numel() for p in model.parameters())
comp = sum(p.numel() for p in compressed.parameters())
print(f"Compression: {orig/comp:.2f}x")
```

> **Note:** Tucker decomposition uses an iterative algorithm (HOOI), so even at `percent_removed=0.0` there will be small reconstruction error. Fine-tuning after decomposition is recommended.

In [ ]:
#| hide
from fastcore.test import *

decomposer = Conv_Decomposer()
_m = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, 32, 3, padding=1))
_x = torch.randn(2, 3, 8, 8)

# === All methods produce correct output shape ===
for method in ['tucker', 'svd', 'spatial', 'cp']:
    _dec = decomposer.decompose(_m, 0.5, method=method)
    test_eq(_m(_x).shape, _dec(_x).shape)
    assert torch.isfinite(_dec(_x)).all(), f"{method} produced non-finite output"

# === Tucker: 3 layers (1x1, KxK, 1x1) ===
_t = decomposer.decompose(_m, 0.5, method='tucker')
test_eq(len(_t[0]), 3)
test_eq(_t[0][0].kernel_size, (1, 1))
test_eq(_t[0][1].kernel_size, (3, 3))

# === SVD: 2 layers (KxK, 1x1) ===
_s = decomposer.decompose(_m, 0.5, method='svd')
test_eq(len(_s[0]), 2)
test_eq(_s[0][0].kernel_size, (3, 3))
test_eq(_s[0][1].kernel_size, (1, 1))

# === Spatial: 2 layers (Kx1, 1xK) ===
_sp = decomposer.decompose(_m, 0.5, method='spatial')
test_eq(len(_sp[0]), 2)
test_eq(_sp[0][0].kernel_size, (3, 1))
test_eq(_sp[0][1].kernel_size, (1, 3))

# === CP: 4 layers (1x1, Kx1, 1xK, 1x1) ===
_cp = decomposer.decompose(_m, 0.5, method='cp')
test_eq(len(_cp[0]), 4)
test_eq(_cp[0][0].kernel_size, (1, 1))  # pointwise in
test_eq(_cp[0][1].kernel_size, (3, 1))  # depthwise vertical
test_eq(_cp[0][2].kernel_size, (1, 3))  # depthwise horizontal
test_eq(_cp[0][3].kernel_size, (1, 1))  # pointwise out

# === Common: 1x1 and grouped skipped ===
assert isinstance(decomposer.decompose(nn.Sequential(nn.Conv2d(16, 32, 1)), 0.5)[0], nn.Conv2d)
assert isinstance(decomposer.decompose(nn.Sequential(nn.Conv2d(16, 16, 3, groups=16, padding=1)), 0.5)[0], nn.Conv2d)

# === Bias: last layer gets it ===
for method in ['tucker', 'svd', 'spatial', 'cp']:
    _dec = decomposer.decompose(nn.Sequential(nn.Conv2d(16, 32, 3, padding=1, bias=True)), 0.5, method=method)
    seq = _dec[0]
    assert seq[-1].bias is not None, f"{method}: last layer missing bias"
    for layer in seq[:-1]:
        assert layer.bias is None, f"{method}: non-last layer has bias"

# === Stride transfer ===
_stride = decomposer.Tucker(nn.Conv2d(16, 32, 3, stride=2, padding=1), 0.5)
test_eq(_stride[1].stride, (2, 2))

_svd_stride = decomposer.SVD(nn.Conv2d(16, 32, 3, stride=2, padding=1), 0.5)
test_eq(_svd_stride[0].stride, (2, 2))

# === Validation ===
with ExceptionExpected(ValueError): decomposer.decompose(nn.Sequential(nn.Conv2d(3, 16, 3)), percent_removed=1.0)
with ExceptionExpected(ValueError): decomposer.decompose(nn.Sequential(nn.Conv2d(3, 16, 3)), method='bad')

# === energy_threshold + layers/exclude ===
_m4 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1))
assert decomposer.decompose(_m4, energy_threshold=0.99)[0][0].out_channels >= \
       decomposer.decompose(_m4, 0.5)[0][0].out_channels

_m5 = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, 32, 3, padding=1))
assert isinstance(decomposer.decompose(_m5, 0.5, layers=['0'])[2], nn.Conv2d)
assert isinstance(decomposer.decompose(_m5, 0.5, exclude=['2'])[2], nn.Conv2d)

In [ ]:
#| hide
#| slow
from torchvision.models import resnet18

# Decompose a real ResNet-18, verify it still works
_resnet = resnet18(weights=None)
_resnet.eval()
_x = torch.randn(2, 3, 64, 64)
_out_orig = _resnet(_x)

_dec = Conv_Decomposer()
_resnet_dec = _dec.decompose(_resnet, percent_removed=0.5)
_resnet_dec.eval()
_out_dec = _resnet_dec(_x)

# Same output shape
test_eq(_out_orig.shape, _out_dec.shape)

# Outputs are finite (no NaN/Inf)
assert torch.isfinite(_out_dec).all(), "Decomposed ResNet produced non-finite outputs"

# Parameter count reduced
_orig_params = sum(p.numel() for p in _resnet.parameters())
_dec_params = sum(p.numel() for p in _resnet_dec.parameters())
assert _dec_params < _orig_params, f"Expected fewer params: {_dec_params} >= {_orig_params}"
print(f"ResNet-18: {_orig_params:,} → {_dec_params:,} params ({_orig_params/_dec_params:.2f}x compression)")

---

## See Also

- [FC Decomposer](fc_decomposer.html) - SVD decomposition for Linear layers
- [BN Folding](bn_folding.html) - Fold BatchNorm into preceding Conv/Linear layers
- [Pruner](../prune/pruner.html) - Structured pruning that removes entire filters